In [2]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/CG_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)

features = ['NDVI', 'NDMI', 'NBR', 'BSI']
df = df[['latitude', 'longitude', 'year'] + features]
df = df.dropna()
df = df.sort_values(by=['latitude', 'longitude', 'year'])

# ===============================
# 3. Create Sequences (RAW DATA)
# ===============================
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

for (lat, lon), group in df.groupby(['latitude', 'longitude']):
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/CG_Deforestation_Predictions.csv',
    index=False
)

print("Saved:CG_Deforestation_Predictions.csv")


Mounted at /content/drive
Raw input shape: (28077, 4, 4)

Label distribution:
No deforestation: 26185
Deforestation: 1892
Train shape: (22461, 4, 4)
Test shape : (5616, 4, 4)
Scaled train shape: (22461, 4, 4)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9048 - loss: 1.1726 - precision: 0.1082 - recall: 0.0828 - val_accuracy: 0.8333 - val_loss: 0.3826 - val_precision: 0.2373 - val_recall: 0.6667
Epoch 2/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.7693 - loss: 0.8519 - precision: 0.1922 - recall: 0.7396 - val_accuracy: 0.9144 - val_loss: 0.2052 - val_precision: 0.4280 - val_recall: 0.8095
Epoch 3/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8746 - loss: 0.5211 - precision: 0.3370 - recall: 0.8692 - val_accuracy: 0.8255 - val_loss: 0.3960 - val_precision: 0.2744 - val_recall: 0.9683
Epoch 4/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8839 - loss: 0.4548 - precision: 0.3637 - recall: 0.9065 - val_accuracy: 0.8709 - val_loss: 0.2873 - val_precision: 0.3392 - val_recall: 0.9683
Epoch 5/20
176/176 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8906 - loss: 0.4192 - precision: 0.3729 - recall: 0.9091 - val_accuracy: 0.9192 - val_loss:

In [3]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/CG_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/CG_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/CG_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.1126356639653453), 'std': 0.07410761074496652, 'q1': np.float64(-0.16675640000000003), 'q3': np.float64(-0.056752170000000046), 'lower_2std': np.float64(-0.26085088545527835), 'upper_2std': np.float64(0.035579557524587746)}
NBR : {'mean': np.float64(-0.06907688503106245), 'std': 0.06564830299297782, 'q1': np.float64(-0.114294), 'q3': np.float64(-0.02644641999999997), 'lower_2std': np.float64(-0.20037349101701807), 'upper_2std': np.float64(0.062219720954893185)}
BSI : {'mean': np.float64(0.0263802567025705), 'std': 0.04124651722091395, 'q1': np.float64(0.002029164533651595), 'q3': np.float64(0.0521808089360777), 'lower_2std': np.float64(-0.0561127777392574), 'upper_2std': np.float64(0.10887329114439839)}
NDMI: {'mean': np.float64(-0.021832313572269477), 'std': 0.04555401689129705, 'q1': np.float64(-0.050411360000000016), 'q3': np.float64(0.004953590000000008), 'lower_2std': np.float64(-0.11294034735486358), 'upper_2std': np.float64(0.06927572021032463)}

Def

In [5]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster


m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/CG_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Logging        537
Agriculture    123
Fire            52
Unknown         39
Urban/Other      8
Name: count, dtype: int64


In [6]:
m.save('/content/drive/MyDrive/cg_adaptive.html')


In [7]:
m

In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/CG_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/CG_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD CHHATTISGARH DISTRICTS
# ==============================

cg = gpd.read_file('/content/drive/MyDrive/CG.geojson')

# CRS SAFETY
if cg.crs is None:
    cg.set_crs("EPSG:4326", inplace=True)

cg = cg.to_crs("EPSG:4326")

# Optional label
cg["state"] = "Chhattisgarh"

print(cg.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 5616
    latitude  longitude  deforestation
0  18.450947  81.419255              0
1  19.166006  81.220727              0
2  19.635825  80.692518              0
3  19.364533  81.058132              0
4  18.914477  81.032979              0
Total deforestation samples: 759
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  18.553355  81.243185              1          NaN         NaN         NaN   
1  22.828437  83.605754              1    -0.155011   -0.140892    0.089279   
2  18.310810  81.278219              1    -0.154001   -0.103150    0.082959   
3  18.759069  81.957346              1    -0.377625   -0.326787    0.169676   
4  18.802188  81.692343              1    -0.252875   -0.169034    0.092236   

   NDMI_change        cause  
0          NaN      Unknown  
1    -0.081304  Agriculture  
2    -0.067562  Agriculture  
3    -0.191281         Fire  
4    -0.091967      Logging  
  REMARKS_2 Country    State_Name State_Code       Dist_Na

In [10]:
# ==============================
# SPATIAL JOIN (Points → Chhattisgarh districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    cg,                 # ← Chhattisgarh district polygons
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — Chhattisgarh
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/CG_District_Wise_Deforestation.csv',
    index=False
)

print("Saved CG district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/CG_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved CG cause summary")

    latitude  longitude  deforestation        cause  \
0  18.553355  81.243185              1      Unknown   
1  22.828437  83.605754              1  Agriculture   
2  18.310810  81.278219              1  Agriculture   
3  18.759069  81.957346              1         Fire   
4  18.802188  81.692343              1      Logging   

                    geometry  index_right REMARKS_2 Country    State_Name  \
0  POINT (81.24319 18.55335)           22      None   India  Chhattisgarh   
1  POINT (83.60575 22.82844)           25      None   India  Chhattisgarh   
2  POINT (81.27822 18.31081)            1      None   India  Chhattisgarh   
3  POINT (81.95735 18.75907)            0      None   India  Chhattisgarh   
4  POINT (81.69234 18.80219)           22      None   India  Chhattisgarh   

  State_Code                 Dist_Name Dist_Code         state  
0         22  Dakshin Bastar Dantewada       416  Chhattisgarh  
1         22                   Surguja       401  Chhattisgarh  
2         2

In [11]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Chhattisgarh District Boundaries
# =====================================================
cg = gpd.read_file('/content/drive/MyDrive/CG.geojson')

if cg.crs is None:
    cg.set_crs(epsg=4326, inplace=True)

cg = cg.to_crs(epsg=4326)
cg["state"] = "Chhattisgarh"
gdf_districts = cg.copy()

# =====================================================
# STEP 3: Load Prediction Data
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/CG_Deforestation_Predictions.csv")

# Ensure required columns exist
if 'deforestation' not in df.columns:
    df['deforestation'] = 0

if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area & Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Map
# =====================================================
m = folium.Map(location=[21.3, 82.0], zoom_start=7)

# =====================================================
# STEP 10: Dynamic Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, 6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Chhattisgarh)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Dynamic Chhattisgarh Alert
# =====================================================
state_total = gdf_districts["total_samples"].sum()
state_def = gdf_districts["deforestation_samples"].sum()
state_aff = gdf_districts["afforestation_samples"].sum()

top3 = gdf_districts.sort_values(
    by="deforestation_samples", ascending=False
).head(3)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

TREE_DENSITY_PER_HA = 1000
total_deforested_area_ha = gdf_districts["deforestation_area_ha"].sum()
trees_to_plant = int(total_deforested_area_ha * TREE_DENSITY_PER_HA)

popup_html = f"""
<b>🌳 Chhattisgarh Forest Alert</b><br><br>

Total Samples: <b>{int(state_total)}</b><br>
Deforestation Points: <b>{int(state_def)}</b><br>
Afforestation Points: <b>{int(state_aff)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

Estimated Trees Required: <b>{trees_to_plant:,}</b><br><br>

🌱 Recommended Actions:<br>
• Tribal community forest programs<br>
• Mixed native species plantation<br>
• Mining area reclamation<br>
• Sustainable land monitoring
"""

folium.Marker(
    location=[21.3, 82.0],
    popup=popup_html,
    icon=folium.Icon(color="darkgreen")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/cg_forest.html")

print("✅ Chhattisgarh map saved successfully with dynamic afforestation alert!")

/tmp/ipython-input-946/1685732326.py:84: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Chhattisgarh map saved successfully with dynamic afforestation alert!
